Step 1: Set Up Google Earth Engine

In [ ]:
!pip install sentinelsat --break-system-packages

In [ ]:
from sentinelsat import SentinelAPI, read_geojson, geojson_to_wkt


### Step 2: The Python Export Script
Sentinel-1 data requires very specific filtering because the satellite captures data in different modes and polarizations. The guide you linked primarily references the COPERNICUS/S1_GRD (Ground Range Detected) collection, which is the standard format for machine learning tasks.

In [ ]:
import requests
import json

# Load your existing area.json
with open('area.geojson', 'r') as f:
    geojson = json.load(f)

# Extract coordinates from your GeoJSON
coords = geojson["features"][0]["geometry"]["coordinates"][0]

# Convert to the bbox string format the API needs
bbox = ", ".join([f"{coord[0]} {coord[1]}" for coord in coords])

# Step 1: Get access token
def get_token(username, password):
    response = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data={
            "client_id": "cdse-public",
            #ADD YOUR EMAIL ADDRESS
            "username": "PUT YOUR EMAIL ADDRESS",
            #ADD YOUR PASSWORD
            "password": "YOUR PASSWORD",
            "grant_type": "password"
        }
    )
    return response.json()["access_token"]

# Step 2: Search for Sentinel-1 products
def search_sentinel1(token, start_date, end_date, bbox):
    headers = {"Authorization": f"Bearer {token}"}

    url = (
        "https://catalogue.dataspace.copernicus.eu/odata/v1/Products?"
        f"$filter=Collection/Name eq 'SENTINEL-1' "
        f"and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' "
        f"and att/OData.CSC.StringAttribute/Value eq 'GRD') "
        f"and ContentDate/Start gt {start_date}T00:00:00.000Z "
        f"and ContentDate/Start lt {end_date}T00:00:00.000Z "
        f"and OData.CSC.Intersects(area=geography'SRID=4326;POLYGON(({bbox}))')"
        f"&$top=10"
    )

    response = requests.get(url, headers=headers)
    return response.json()

# Step 3: Download a product
def download_product(token, product_id, filename):
    headers = {"Authorization": f"Bearer {token}"}
    url = f"https://download.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"

    response = requests.get(url, headers=headers, stream=True)
    with open(filename, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded: {filename}")

# --- Run it ---
#ADD YOUR EMAIL ADDRESS
USERNAME = ""
#ADD YOUR PASSWORD
PASSWORD = ""

token = get_token(USERNAME, PASSWORD)
print("Token obtained successfully")

results = search_sentinel1(token, "2023-01-01", "2023-03-01", bbox)
products = results["value"]
print(f"Found {len(products)} products")

# Download first result only (test first before bulk downloading)
if products:
    product_id = products[0]["Id"]
    name = products[0]["Name"]
    print(f"Downloading: {name}")
    download_product(token, product_id, f"{name}.zip")
else:
    print("No products found")